In [ ]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.avro.functions import from_avro

In [ ]:
spark = SparkSession.builder.appName("Kafka Example").getOrCreate()
hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.access.key", "mmix")
hconf.set("fs.s3a.secret.key", "mmixmmix")
hconf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hconf.set("fs.s3a.path.style.access", "false")

In [ ]:
dataframe = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "confluent-kafka-broker.mmix.io:9093") \
    .option("kafka.security.protocol", "SASL_PLAINTEXT") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", "org.apache.kafka.common.security.plain.PlainLoginModule ""required username=admin password=admin;") \
    .option("subscribe", "") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

### Print to Console

In [ ]:
q = dataframe \
    .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value") \
    .writeStream.format("console").option("truncate", "false") \
    .trigger(processingTime="5 seconds") \
    .start()
q.awaitTermination()

### Non Avro Schema

In [ ]:
q = dataframe \
    .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "timestamp") \
    .writeStream \
    .format("parquet") \
    .option("path", "s3a://mmix-prod-dataengineer-datalakehouse/reducing/stream/mmix/products/") \
    .option("checkpointLocation", "s3a://mmix-prod-dataengineer-datalakehouse/reducing/stream/spark-checkpoints/product/") \
    .option("compression", "snappy") \
    .trigger(processingTime="10 seconds") \
    .start()

q.awaitTermination()

### Avro Schema

In [ ]:
schema_registry_url = "http://confluent-schema-registry.mmix.io:8081"
subject = "mmix-event-logs-topic-avro-value"
schema_str = requests.get(f"{schema_registry_url}/subjects/{subject}/versions/latest").json()["schema"]

In [ ]:
q = dataframe \
    .select(from_avro(col("value"), schema_str, {"mode": "PERMISSIVE"}).alias("data")) \
    .writeStream \
    .format("parquet") \
    .option("compression", "zstd") \
    .option("path", "s3a://mmix-prod-dataengineer-datalakehouse/reducing/stream/avro/mmix/event_logs/") \
    .option("checkpointLocation", "s3a://mmix-prod-dataengineer-datalakehouse/reducing/stream/spark-checkpoints/event_logs/") \
    .trigger(processingTime="300 seconds") \
    .start()

q.awaitTermination()